# CST4265 Hackathon - Car Brochure RAG System

## Overview
This notebook implements a complete Retrieval-Augmented Generation (RAG) pipeline for analysing automotive brochure documents and generating grounded responses to user queries.

The implementation follows the lecture-style structure used throughout the CST4265 module. It includes environment setup, provider-based LLM configuration, component tester blocks, brochure upload, PDF loading, chunking, embedding generation, FAISS vector storage, semantic retrieval, structured prompt design, and final grounded response generation.

### This notebook covers:
1. Setting up the environment and installing required packages
2. Importing libraries and configuring the project
3. Selecting and initializing an LLM provider using a switch-based setup
4. Initializing and validating:
   - the LLM
   - the embedding model
   - the vector store
5. Uploading and loading a car brochure PDF
6. Previewing and chunking the brochure text
7. Filtering low-value brochure chunks
8. Creating embeddings and a FAISS vector store
9. Building a RAG pipeline for brochure question answering
10. Applying a lightweight reliability mechanism using retrieval score thresholding
11. Demonstrating the system with factual, feature-based, and interpretive queries

## Project Summary

**Problem Statement:** Car brochures contain useful specifications, features, and descriptive information, but users must manually search long documents to find relevant details.

**Value Proposition:** This system allows a user or lecturer to upload a brochure and ask targeted questions, receiving structured answers grounded in the brochure itself.

**AI Component:** The system uses a Retrieval-Augmented Generation (RAG) pipeline to retrieve relevant brochure chunks and generate responses using only the retrieved context.

## Key Concepts Demonstrated
- Context Engineering (Lecture 2)
- Embeddings and Vector Stores (Lecture 4)
- Retrieval-Augmented Generation (Lecture 5)
- LLMOps / Demo-to-System Thinking (Lecture 8)
- Notebook-based modular GenAI system design

## Notes
- This notebook is designed for **Google Colab**
- Tester blocks are included immediately after component initialization
- The LLM provider can be changed through a single configuration variable
- The embedding model is local and uses HuggingFace
- The system prioritizes grounded answers over unsupported generation

## Intended Outcome
By the end of this notebook, the system will be able to:
- upload and process a car brochure PDF
- convert brochure text into retrievable chunks
- retrieve relevant evidence for a user query
- generate structured and brochure-grounded answers

In [1]:
# source_handbook: week11-hackathon-preparation

## How to Run

1. Run the setup cells in Section 1.
2. Enter the API key for the selected provider when prompted.
3. Upload a brochure PDF in Section 2.
4. Run the preprocessing and vector store sections.
5. Run the example queries in Section 5.
6. Optionally, enter your own question in Section 6.

## Environment Note

This notebook is designed primarily for Google Colab. The brochure upload functionality uses the Colab file upload interface.

# 1. Setup and Initialization

## 1.1 Package Installation

This section installs the core packages required for the RAG pipeline. The setup follows the practical notebook style used in the lecture materials, where package installation is completed before initialization and validation.

In [2]:
# 1.1 Package Installation

!pip install -q langchain langchain-core langchain-community
!pip install -q langchain-openai langchain-google-genai langchain-groq
!pip install -q faiss-cpu sentence-transformers langchain-huggingface
!pip install -q pypdf

## 1.2 Optional Package Verification

This section performs a quick verification of key installed packages before the main notebook pipeline is executed.

In [3]:
# 1.2 Optional Package Verification

!pip list | grep -Ei "langchain|faiss|sentence|pypdf|groq|openai|google-genai"

faiss-cpu                                1.13.2
google-genai                             1.68.0
groq                                     0.37.1
langchain                                1.2.15
langchain-classic                        1.0.4
langchain-community                      0.4.1
langchain-core                           1.3.0
langchain-google-genai                   4.2.2
langchain-groq                           1.1.2
langchain-huggingface                    1.2.2
langchain-openai                         1.2.0
langchain-text-splitters                 1.1.2
openai                                   2.31.0
pypdf                                    6.10.2
sentence-transformers                    5.4.0
sentencepiece                            0.2.1


## 1.3 Library Imports

This section imports the libraries used throughout the notebook. The structure separates prompt/pipeline components, provider-specific LLM classes, and retrieval-related modules.

In [4]:
# 1.3 Library Imports

import os
import textwrap
from getpass import getpass

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

## 1.4 Project Configuration

This section defines the core system parameters used throughout the notebook. The retrieval threshold is tuned to balance precision and recall, allowing the system to answer both factual and interpretive brochure queries while still preventing weak-evidence answers.

Chunk size is set to preserve more coherent brochure context, and retrieval settings are chosen to improve result diversity.

In [5]:
# 1.4 Project Configuration

API_KEY_PROVIDER = "GROQ"   # Options: "GROQ", "OPENAI", "GEMINI"
TEMPERATURE = 0.2

# Retrieval settings
TOP_K = 4
RETRIEVAL_SCORE_THRESHOLD = 1.4

# Chunking settings
CHUNK_SIZE = 700
CHUNK_OVERLAP = 120

# Embedding model
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

## 1.5 API Key Setup

This section loads the API key for the selected provider. The provider can be changed by editing the `API_KEY_PROVIDER` variable in the configuration section.

In [6]:
# 1.5 API Key Setup

def ensure_api_key(provider: str):
    if provider == "GROQ":
        env_name = "GROQ_API_KEY"
    elif provider == "OPENAI":
        env_name = "OPENAI_API_KEY"
    elif provider == "GEMINI":
        env_name = "GOOGLE_API_KEY"
    else:
        raise ValueError("Unsupported provider")

    if not os.environ.get(env_name):
        os.environ[env_name] = getpass(f"Enter {env_name}: ")

    return env_name

active_env_name = ensure_api_key(API_KEY_PROVIDER)
print(f"Active provider: {API_KEY_PROVIDER}")
print(f"Loaded key from environment variable: {active_env_name}")

Enter GROQ_API_KEY: ··········
Active provider: GROQ
Loaded key from environment variable: GROQ_API_KEY


## 1.6 LLM Factory Function (Final Fix)

This section initializes the LLM using the selected provider.

The Groq model has been updated to a currently supported model (`llama-3.1-8b-instant`) following recent deprecations. This ensures compatibility with the Groq API and stable execution of the RAG pipeline.

In [9]:
# 1.6 LLM Factory Function (Final Fix - Working Groq Models)

def get_llm(provider: str = "GROQ", temperature: float = 0.2):
    provider = provider.upper()

    if provider == "GROQ":
        return ChatGroq(
            model_name="llama-3.1-8b-instant",   # ✅ WORKING MODEL
            temperature=temperature
        )

    elif provider == "OPENAI":
        return ChatOpenAI(
            model="gpt-4o-mini",
            temperature=temperature
        )

    elif provider == "GEMINI":
        return ChatGoogleGenerativeAI(
            model="gemini-1.5-flash",
            temperature=temperature
        )

    else:
        raise ValueError("Unsupported provider selected.")

llm = get_llm(API_KEY_PROVIDER, TEMPERATURE)
print("LLM initialized successfully.")

LLM initialized successfully.


## 1.7 Tester Blocks

This section validates the core components immediately after initialization. The notebook tests:
- the LLM
- the embedding model
- a small FAISS vector store

This reflects the lecture-style engineering practice of validating components before building the complete system.

In [10]:
# 1.7 Tester Blocks

# ---- LLM Tester ----
print("Testing LLM...")
try:
    llm_test = llm.invoke("Reply ONLY with: LLM working")
    print("LLM test output:", llm_test.content if hasattr(llm_test, "content") else llm_test)
except Exception as e:
    print("LLM test failed:", e)

# ---- Embedding Model Initialization ----
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME
)

# ---- Embedding Tester ----
print("\nTesting embeddings...")
try:
    test_vector = embeddings.embed_query("car engine horsepower")
    print("Embedding length:", len(test_vector))
except Exception as e:
    print("Embedding test failed:", e)

# ---- Vector Store Tester ----
print("\nTesting vector store...")
try:
    sample_texts = [
        "The vehicle uses a 2.0L turbocharged petrol engine with 250 horsepower.",
        "Fuel consumption is 6.5 litres per 100 kilometres on the combined cycle.",
        "The infotainment system supports Apple CarPlay and Android Auto."
    ]
    sample_metadatas = [
        {"source": "tester", "page": 1},
        {"source": "tester", "page": 1},
        {"source": "tester", "page": 2},
    ]

    test_vectorstore = FAISS.from_texts(
        texts=sample_texts,
        embedding=embeddings,
        metadatas=sample_metadatas
    )

    test_results = test_vectorstore.similarity_search("engine power", k=1)
    print("Vector store retrieval output:", test_results[0].page_content)
except Exception as e:
    print("Vector store test failed:", e)

Testing LLM...
LLM test output: LLM working


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Testing embeddings...
Embedding length: 384

Testing vector store...
Vector store retrieval output: The vehicle uses a 2.0L turbocharged petrol engine with 250 horsepower.


# 2. Document Loading and Preprocessing

## 2.1 User Upload of Car Brochure PDF

This section allows the user or lecturer to upload any automotive brochure PDF directly into the notebook environment. The uploaded file becomes the source document for the remaining RAG pipeline stages.

In [11]:
# 2.1 User Upload of Car Brochure PDF

from google.colab import files

print("Please upload a car brochure PDF file.")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No file was uploaded. Please upload a PDF brochure file.")

uploaded_filename = list(uploaded.keys())[0]

if not uploaded_filename.lower().endswith(".pdf"):
    raise ValueError("Invalid file type. Please upload a PDF file only.")

print(f"Uploaded file: {uploaded_filename}")

Please upload a car brochure PDF file.


Saving Toyota-GR86-2026-ZA.pdf to Toyota-GR86-2026-ZA (2).pdf
Uploaded file: Toyota-GR86-2026-ZA (2).pdf


## 2.2 Load the Uploaded PDF Brochure

This section loads the uploaded brochure using `PyPDFLoader`. A validation check is applied to ensure that the document contains readable pages before the pipeline proceeds.

In [12]:
# 2.2 Load the Uploaded PDF Brochure

loader = PyPDFLoader(uploaded_filename)
documents = loader.load()

if len(documents) == 0:
    raise ValueError("❌ The uploaded PDF contains no readable pages or failed to load.")

print(f"✅ Total pages loaded: {len(documents)}")
print("\nSample metadata from first page:")
print(documents[0].metadata)

✅ Total pages loaded: 9

Sample metadata from first page:
{'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.1 (Macintosh)', 'creationdate': '2026-01-28T10:53:30+02:00', 'moddate': '2026-01-28T10:53:42+02:00', 'trapped': '/False', 'source': 'Toyota-GR86-2026-ZA (2).pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}


## 2.3 Preview Raw Document Content

This section previews a portion of the extracted text from the first page of the uploaded brochure. A lightweight check is used to confirm that readable text exists.

In [13]:
# 2.3 Preview Raw Document Content

if len(documents) > 0:
    first_page_text = documents[0].page_content.strip()

    if not first_page_text:
        print("⚠️ Warning: First page contains no readable text.")
    else:
        print("✅ Preview of extracted text:\n")
        print(textwrap.fill(first_page_text[:1000], width=100))
else:
    print("⚠️ No documents available for preview.")

✅ Preview of extracted text:

86


## 2.4 Chunk the Brochure Text

This section uses `RecursiveCharacterTextSplitter` to divide the brochure into overlapping text chunks. This improves retrieval quality by allowing the system to work with focused local context rather than the full document at once.

In [14]:
# 2.4 Chunk the Brochure Text

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

chunks = text_splitter.split_documents(documents)

if len(chunks) == 0:
    raise ValueError("❌ No chunks were created from the brochure. The document may be empty or unreadable.")

print(f"✅ Total chunks created: {len(chunks)}")
print("\nSample chunk metadata:")
print(chunks[0].metadata)
print("\nSample chunk content:")
print(textwrap.fill(chunks[0].page_content[:800], width=100))

✅ Total chunks created: 33

Sample chunk metadata:
{'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.1 (Macintosh)', 'creationdate': '2026-01-28T10:53:30+02:00', 'moddate': '2026-01-28T10:53:42+02:00', 'trapped': '/False', 'source': 'Toyota-GR86-2026-ZA (2).pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}

Sample chunk content:
86


## 2.5 Filter Low-Value Brochure Chunks

This section removes low-value brochure content such as legal disclaimers and financial-service material. These sections are unlikely to help with vehicle-focused question answering and can reduce retrieval quality if kept in the vector store.

In [15]:
# 2.5 Filter Low-Value Brochure Chunks

LOW_VALUE_KEYWORDS = [
    "DISCLAIMER",
    "FINANCIAL SERVICES",
    "REGISTERED CREDIT PROVIDER",
    "SUBJECT TO CHANGE WITHOUT NOTICE",
    "YOUR USE OF, OR RELIANCE ON",
    "TERMS AND CONDITIONS",
    "TS & CS APPLY"
]

filtered_chunks = []

for chunk in chunks:
    text_upper = chunk.page_content.upper()

    if not any(keyword in text_upper for keyword in LOW_VALUE_KEYWORDS):
        filtered_chunks.append(chunk)

if len(filtered_chunks) == 0:
    raise ValueError("❌ All chunks were filtered out.")

print(f"Original chunks: {len(chunks)}")
print(f"Filtered chunks: {len(filtered_chunks)}")

chunks = filtered_chunks

Original chunks: 33
Filtered chunks: 28


# 3. Embeddings and Vector Store Construction

## 3.1 Create the FAISS Vector Store

This section converts the brochure chunks into embeddings and stores them in a FAISS vector database. This follows the local embedding + vector store pattern used in the lecture notebooks.

In [16]:
# 3.1 Create the FAISS Vector Store

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("FAISS vector store created successfully.")

FAISS vector store created successfully.


## 3.2 Create the Retriever

This retriever uses Maximum Marginal Relevance (MMR) to improve diversity among retrieved results. This reduces redundancy and helps broader queries retrieve complementary information from different parts of the brochure.

In [17]:
# 3.2 Create the Retriever

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": TOP_K,
        "fetch_k": 8
    }
)

print("Retriever (MMR) initialized.")

Retriever (MMR) initialized.


## 3.3 Retrieval Test on the Real Brochure

This section performs a score-aware retrieval test on the uploaded brochure before full answer generation begins. Displaying scored retrieval results makes the system easier to inspect during marking and demonstration.

In [18]:
# 3.3 Retrieval Test on the Real Brochure

retrieval_test_query = "What does the brochure say about engine performance?"

retrieval_test_results = vectorstore.similarity_search_with_score(
    retrieval_test_query,
    k=TOP_K
)

print(f"Retrieved {len(retrieval_test_results)} scored chunks for test query.\n")

for i, (doc, score) in enumerate(retrieval_test_results, start=1):
    print(f"--- Retrieved Chunk {i} ---")
    print(f"Score: {score:.4f}")
    print("Metadata:", doc.metadata)
    print(textwrap.fill(doc.page_content[:500], width=100))
    print()

Retrieved 4 scored chunks for test query.

--- Retrieved Chunk 1 ---
Score: 1.0118
Metadata: {'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.1 (Macintosh)', 'creationdate': '2026-01-28T10:53:30+02:00', 'moddate': '2026-01-28T10:53:42+02:00', 'trapped': '/False', 'source': 'Toyota-GR86-2026-ZA (2).pdf', 'total_pages': 9, 'page': 1, 'page_label': '2'}
powerful 2.4ℓ boxer engine that punches out more kilowatts and more  bottom-end torque than ever
before, whether powering off the line or  out of your favourite hairpin corner. THE GR86 IS BORN TO
PERFORM. MODEL SHOWN: GR86 AT THE ICON,  UNTAMED

--- Retrieved Chunk 2 ---
Score: 1.1378
Metadata: {'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.1 (Macintosh)', 'creationdate': '2026-01-28T10:53:30+02:00', 'moddate': '2026-01-28T10:53:42+02:00', 'trapped': '/False', 'source': 'Toyota-GR86-2026-ZA (2).pdf', 'total_pages': 9, 'page': 3, 'page_label': '4'}
ALL TORQUE ALL ACTION Its larger capacity, natural

# 4. Prompt and RAG Pipeline Construction

## 4.1 Prompt Template

This prompt enforces grounded responses using only brochure evidence while allowing cautious interpretation for broader or user-focused queries.

Additional constraints are included to ensure:
- answers remain strictly supported by retrieved context
- summaries include meaningful technical or descriptive details
- hallucination and unsupported feature generation are prevented

The structured output format improves readability, consistency, and evaluation during marking.

In [19]:
# 4.1 Prompt Template (Final)

rag_prompt = ChatPromptTemplate.from_template("""
You are an automotive brochure assistant.

Your task is to answer the user's question using ONLY the provided brochure context.

If the context is insufficient, unclear, or not relevant enough to answer the question confidently,
say exactly:
"Insufficient brochure evidence to answer this question reliably."

Do not invent specifications, features, performance data, safety features, or recommendations
that are not supported by the brochure.

Only include details that are explicitly supported by the provided context.
Avoid introducing features, specifications, or statements that are not clearly present in the retrieved text.

If the question is interpretive (for example, asking what type of driver the car may suit),
you may make a cautious interpretation, but only if it is clearly supported by the brochure language.

Return the answer in this exact format:

1. Summary
- The summary must include key technical or descriptive details where available.
- Avoid generic phrasing.

2. Key Details from the Brochure
- Use bullet points.
- Include exact brochure-supported details where possible.

3. Recommendation / Interpretation
- Keep this grounded in the brochure.
- Do not make unsupported claims.

Brochure Context:
{context}

User Question:
{question}
""")

## 4.2 Utility Functions and Reliability Controls

This section defines the helper functions used for retrieval, context formatting, score inspection, and final answer generation.

A simple reliability mechanism is included. If retrieved evidence is too weak, the system abstains rather than generating an unsupported answer. This threshold acts as a lightweight safeguard rather than a formal confidence guarantee.

In [20]:
# 4.2 Utility Functions and Reliability Controls

def format_docs(docs):
    """
    Convert retrieved brochure chunks into a readable context block.
    """
    formatted_parts = []

    for i, doc in enumerate(docs, start=1):
        page = doc.metadata.get("page", "unknown")
        source = doc.metadata.get("source", "unknown")
        content = doc.page_content.strip()

        formatted_parts.append(
            f"[Chunk {i} | Source: {source} | Page: {page}]\n{content}"
        )

    return "\n\n".join(formatted_parts)


def retrieve_with_scores(query, k=TOP_K):
    """
    Retrieve the top-k most relevant brochure chunks with similarity scores.
    Lower scores are typically better for FAISS distance-based retrieval.
    """
    return vectorstore.similarity_search_with_score(query, k=k)


def is_retrieval_strong(scored_results, threshold=RETRIEVAL_SCORE_THRESHOLD):
    """
    Determine whether retrieval is strong enough to proceed with answer generation.
    """
    if not scored_results:
        return False

    best_score = scored_results[0][1]
    return best_score <= threshold


def build_context_from_scored_results(scored_results):
    """
    Build the final context string from scored retrieval results.
    """
    docs_only = [doc for doc, score in scored_results]
    return format_docs(docs_only)


def answer_brochure_query(query, threshold=RETRIEVAL_SCORE_THRESHOLD, k=TOP_K, show_debug=True):
    """
    End-to-end RAG function for brochure question answering.
    """
    scored_results = retrieve_with_scores(query, k=k)

    if show_debug:
        print(f"Top {len(scored_results)} retrieved results:\n")
        for idx, (doc, score) in enumerate(scored_results, start=1):
            print(f"--- Result {idx} ---")
            print(f"Score: {score:.4f}")
            print(f"Metadata: {doc.metadata}")
            print(textwrap.fill(doc.page_content[:500], width=100))
            print()

    if not is_retrieval_strong(scored_results, threshold=threshold):
        return {
            "status": "abstain",
            "answer": "Insufficient brochure evidence to answer this question reliably.",
            "context": "",
            "results": scored_results
        }

    context = build_context_from_scored_results(scored_results)

    chain = rag_prompt | llm | StrOutputParser()

    final_answer = chain.invoke({
        "context": context,
        "question": query
    })

    return {
        "status": "answered",
        "answer": final_answer,
        "context": context,
        "results": scored_results
    }

# 5. Query Demonstration

These queries are designed to test three different types of system behavior:

- a factual performance query
- a feature-based brochure query
- an interpretive driver-profile query

The example queries and retrieved-context outputs provide direct evidence of system behavior and grounding.

In [21]:
# 5. Query Demonstration

query_1 = "What does the brochure say about the engine performance of this car?"

query_2 = "What interior technology and driver features are included in the GR86?"

query_3 = "What does the brochure suggest about the type of driver this car is designed for?"

## 5.1 Example Query 1

This section runs a factual brochure question through the complete RAG pipeline. The query is designed to test whether the system can retrieve explicit engine specifications and technical performance details.

In [22]:
# 5.1 Example Query 1

response_1 = answer_brochure_query(query_1, show_debug=True)

print("\nFinal System Output:\n")
print(response_1["answer"])

Top 4 retrieved results:

--- Result 1 ---
Score: 0.9968
Metadata: {'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.1 (Macintosh)', 'creationdate': '2026-01-28T10:53:30+02:00', 'moddate': '2026-01-28T10:53:42+02:00', 'trapped': '/False', 'source': 'Toyota-GR86-2026-ZA (2).pdf', 'total_pages': 9, 'page': 1, 'page_label': '2'}
powerful 2.4ℓ boxer engine that punches out more kilowatts and more  bottom-end torque than ever
before, whether powering off the line or  out of your favourite hairpin corner. THE GR86 IS BORN TO
PERFORM. MODEL SHOWN: GR86 AT THE ICON,  UNTAMED

--- Result 2 ---
Score: 1.1297
Metadata: {'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.1 (Macintosh)', 'creationdate': '2026-01-28T10:53:30+02:00', 'moddate': '2026-01-28T10:53:42+02:00', 'trapped': '/False', 'source': 'Toyota-GR86-2026-ZA (2).pdf', 'total_pages': 9, 'page': 1, 'page_label': '2'}
INTRODUCTION The second-generation GR86 is crafted to  turn heads. Born from T oyota’

## 5.2 Example Query 2

This section runs a broader feature-oriented query through the RAG pipeline. The purpose is to test whether the system can retrieve brochure evidence related to interior technology and driver-oriented features.

In [23]:
# 5.2 Example Query 2

response_2 = answer_brochure_query(query_2, show_debug=True)

print("\nFinal System Output:\n")
print(response_2["answer"])

Top 4 retrieved results:

--- Result 1 ---
Score: 0.5778
Metadata: {'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.1 (Macintosh)', 'creationdate': '2026-01-28T10:53:30+02:00', 'moddate': '2026-01-28T10:53:42+02:00', 'trapped': '/False', 'source': 'Toyota-GR86-2026-ZA (2).pdf', 'total_pages': 9, 'page': 4, 'page_label': '5'}
is the proud placement of  GR badges on the front   grille and boot lid. Similar to Toyota’s Le
Mans-winning  endurance race cars, to reduce drag,  lift and add steering stability, the GR86
features air outlets behind the front  wheels to allow air trapped in the wheel  arch to escape.
Specially mounted  fins correspondingly condition airflow  around the rear wheel arches. The GR86
also features lightweight, matte black,  innovatively-styled 18” alloy wheels to  make sure you look
good while y

--- Result 2 ---
Score: 0.6226
Metadata: {'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.1 (Macintosh)', 'creationdate': '2026-01-28

## 5.3 Example Query 3

This section runs an interpretive query through the pipeline. It evaluates whether the system can use brochure language to make a cautious, grounded judgment about the type of driver the car is intended for.

In [24]:
# 5.3 Example Query 3

response_3 = answer_brochure_query(query_3, show_debug=True)

print("\nFinal System Output:\n")
print(response_3["answer"])

Top 4 retrieved results:

--- Result 1 ---
Score: 1.0724
Metadata: {'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.1 (Macintosh)', 'creationdate': '2026-01-28T10:53:30+02:00', 'moddate': '2026-01-28T10:53:42+02:00', 'trapped': '/False', 'source': 'Toyota-GR86-2026-ZA (2).pdf', 'total_pages': 9, 'page': 1, 'page_label': '2'}
INTRODUCTION The second-generation GR86 is crafted to  turn heads. Born from T oyota’s supreme
legacy of sports coupés, this lightweight,  perfectly-balanced, rear-wheel driven beast   is built
to thrill. Like all of Toyota’s GR models, the GR86 is inspired by Toyota Gazoo  Racing, a
motorsports brand that’s conquered some of the world’s  toughest races, including Dakar and the 24
Hours of Le Mans.  Applying engineering solutions mastered in the world’s toughest  test conditions
– the racetrack

--- Result 2 ---
Score: 1.1627
Metadata: {'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.1 (Macintosh)', 'creationdate': '2026-01-2

# 6. Optional Interactive Query Cell

## 6.1 User Query Input

This optional section allows the notebook to function as a lightweight interactive system. A user or lecturer can enter a custom brochure question and observe both retrieval behavior and the final grounded answer.

In [25]:
# 6.1 User Query Input

user_query = input("Enter your brochure question: ").strip()

user_response = answer_brochure_query(user_query, show_debug=True)

print("\nFinal System Output:\n")
print(user_response["answer"])

Enter your brochure question: Give the car's power figure.
Top 4 retrieved results:

--- Result 1 ---
Score: 1.3687
Metadata: {'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.1 (Macintosh)', 'creationdate': '2026-01-28T10:53:30+02:00', 'moddate': '2026-01-28T10:53:42+02:00', 'trapped': '/False', 'source': 'Toyota-GR86-2026-ZA (2).pdf', 'total_pages': 9, 'page': 7, 'page_label': '8'}
Vehicle specifications may vary depending on the model, trim level, and market.  Colours depicted on
print and web material may not match the actual vehicle colour.  Fuel consumption figures are
estimates based on standardised testing conditions and  actual results may vary due to (but not
limited to) driving style, vehicle loading, tyres,  wind, traffic, and accessories. All dimensions,
capacities, and performance figures  are approximate. Some features/options may be optional or only
available

--- Result 2 ---
Score: 1.4360
Metadata: {'producer': 'Adobe PDF Library 18.0', 'creator': '

# 7. Optional Evidence Inspection

## 7.1 Display Retrieved Context for the Last Query

This section displays the exact context used for the most recent answer. It provides direct evidence that the final output is based on brochure retrieval rather than unsupported generation.

In [26]:
# 7.1 Display Retrieved Context for the Last Query

if "user_response" in globals():
    print("Retrieved Context Used:\n")
    print(user_response["context"])
else:
    print("Run Section 6.1 first to inspect the retrieved context.")

Retrieved Context Used:

[Chunk 1 | Source: Toyota-GR86-2026-ZA (2).pdf | Page: 7]
Vehicle specifications may vary depending on the model, trim level, and market. 
Colours depicted on print and web material may not match the actual vehicle colour. 
Fuel consumption figures are estimates based on standardised testing conditions and 
actual results may vary due to (but not limited to) driving style, vehicle loading, tyres, 
wind, traffic, and accessories. All dimensions, capacities, and performance figures 
are approximate. Some features/options may be optional or only available on certain 
models/trim levels. Images and illustrations are for illustrative purposes only and may 
not reflect the actual vehicle.

[Chunk 2 | Source: Toyota-GR86-2026-ZA (2).pdf | Page: 7]
CHALLENGE OF MINIMISING AND OPTIMISING  
WATER USAGE
CHALLENGE 4
By owning a Toyota you would be contributing to the 
reduction of vehicle CO2 emissions by 90% in 2050 
as opposed to 2010. You will help us promote the 
devel

# 8. Conclusion

This notebook implemented a complete notebook-based RAG system for automotive brochure analysis. The final workflow includes setup validation, brochure upload, PDF loading, chunking, filtering, embeddings, vector retrieval, structured prompting, and grounded answer generation.

The resulting system demonstrates the core concepts of context engineering, semantic retrieval, Retrieval-Augmented Generation, and lightweight LLMOps reliability controls in a focused hackathon-appropriate form.

## Future Improvement: Evaluation

A future improvement would involve evaluating retrieval quality and answer faithfulness using a fixed set of brochure-based test questions. This would make the system easier to compare systematically across multiple brochure documents.

## Final Remarks

This notebook implements a minimal but reliable hackathon-ready RAG system. It prioritizes grounded retrieval, scope discipline, modular design, and clear system behavior over unnecessary complexity, making it suitable for both demonstration and marking.